# Gold Apple Ultimate Scraper
Обновленная версия: все настройки и код синхронизированы.

In [1]:
import asyncio
import csv
import json
import logging
import os
import random
import re
import sys
import time
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright
from config import (
    HEADERS, FETCH_DETAILS, DELAY_MIN, DELAY_MAX, 
    CATALOG_URL, PRODUCT_BASE_URL, OUTPUT_FILE
)

# Configuration for the ULTIMATE SCRAPER


## 1. Конфигурация
Здесь можно настроить файл вывода, задержки и стартовую страницу.

In [2]:
CONCURRENCY_CATALOG = 1 # We process catalog pages sequentially to keep state, but can be parallelized if needed
CONCURRENCY_DETAILS = 5 # Reduced concurrency to lower load (requested by user)
MAX_PAGES = 920 # ~22k products / 24 per page

# Setup logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.FileHandler("scraper.log", encoding="utf-8"),
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)



## 2. Логика Скрапера

In [3]:
class GoldAppleUltimateScraper:
    def __init__(self):
        # Инициализация путей, полей CSV и запуск создания файла
        self.output_file = OUTPUT_FILE
        self.fieldnames = [
            "itemId", "brand", "name", "price", "link", "rating", "reviews_count",
            "description", "application", "composition", "brand_info",
            "product_type", "for_whom", "purpose", "skin_type", "area",
            "active_ingredient", "application_time", "volume"
        ]
        self._init_csv()

    def _init_csv(self):
        """Initializes CSV file or skips if resuming."""
        if os.path.exists(self.output_file) and os.path.getsize(self.output_file) > 0:
            logger.info(f"Resuming with existing CSV: {self.output_file}")
            return
        with open(self.output_file, mode="w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames)
            writer.writeheader()
        logger.info(f"Initialized fresh CSV: {self.output_file}")

    def get_last_processed_page(self):
        # Определяет, на какой странице остановились, исходя из количества строк в CSV
        if not os.path.exists(self.output_file): return 0
        try:
            # Считаем только уникальные itemId, если это возможно, либо просто непустые строки
            with open(self.output_file, 'r', encoding='utf-8', errors='replace') as f:
                # Читаем через csv для корректности
                reader = csv.reader(f)
                next(reader, None) # skip header
                count = sum(1 for row in reader if row and any(row))
                
                # ~24 товара на страницу
                page = count // 24
                logger.info(f"Found {count} existing items. Resuming from page {page + 1}")
                return page
        except Exception as e:
            logger.error(f"Error reading {self.output_file} for resume: {e}")
            return 0

    async def get_total_pages(self, browser):
        # Загружает первую страницу и парсит общее количество страниц в пагинации
        context = await browser.new_context(user_agent=HEADERS['user-agent'])
        page = await context.new_page()
        try:
            await page.goto(f"{CATALOG_URL}?p=1", wait_until="domcontentloaded", timeout=60000)
            html = await page.content()
            soup = BeautifulSoup(html, "lxml")
            pagination = soup.select("li[class*='pagination']")
            if pagination:
                pages = [int(p.get_text(strip=True)) for p in pagination if p.get_text(strip=True).isdigit()]
                return max(pages) if pages else 540
            return 900
        except:
            return 900
        finally:
            await context.close()

    async def scrape_catalog_page(self, context, page_num):
        # Загружает страницу каталога и прокручивает её для подгрузки всех товаров
        page = await context.new_page()
        try:
            # Try different pagination parameters
            urls = [
                f"{CATALOG_URL}?p={page_num}",
                f"{CATALOG_URL}?pageNumber={page_num}",
                f"{CATALOG_URL}/page/{page_num}"
            ]
            products = []
            for url in urls:
                logger.info(f"Indexing Catalog: Page {page_num} via {url}")
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                
                # Dynamic wait loop for items to hydrate
                for attempt in range(1, 15): # Increased attempts
                    await page.mouse.wheel(0, 500)
                    await asyncio.sleep(1.5)
                    
                    html = await page.content()
                    products = self._parse_catalog_html(html)
                    logger.info(f"  Page {page_num} attempt {attempt}: Found {len(products)} products (HTML len: {len(html)})")
                    if len(products) >= 10: break # Found enough items
                
                if products: break
            
            if not products:
                logger.warning(f"Page {page_num} is empty or failed.")
            return products
        except Exception as e:
            logger.error(f"Error on catalog page {page_num}: {e}")
            return []
        finally:
            await page.close()

    def _parse_catalog_html(self, html):
        # Извлекает базовые данные о товарах (ID, ссылка, название, цена) из HTML каталога
        soup = BeautifulSoup(html, "lxml")
        products = []
        cards = soup.select("div[class*='_ga-product-card-vertical__container']") or \
                soup.select("div[class*='_ga-product-card-horizontal__container']") or \
                soup.select("article") or \
                soup.select("a[class*='product-card']")
        
        for card in cards:
            try:
                sku_tag = card.select_one('meta[itemprop="sku"]')
                item_id = sku_tag['content'] if sku_tag else card.get("data-product-id")
                
                link_tag = card.select_one('a[class*="__link"]') or \
                           card.select_one('a[class*="product-card"]') or \
                           card if card.name == 'a' else card.select_one("a[href*='/']")
                
                url = link_tag["href"] if link_tag else ""
                if url and not url.startswith("http"):
                    url = f"{PRODUCT_BASE_URL}{url}"

                if not item_id and url:
                    sku_match = re.search(r'/(\d+)-', url)
                    if sku_match: item_id = sku_match.group(1)

                if not item_id: continue

                brand_el = card.select_one("[class*='brand'], [itemprop='brand']")
                brand = brand_el.get_text(strip=True) if brand_el else ""
                
                name_el = card.select_one("span[class*='name']")
                name = name_el.get_text(strip=True) if name_el else ""
                
                price_el = card.select_one("[class*='price']")
                price = "".join(filter(str.isdigit, price_el.get_text())) if price_el else ""

                products.append({
                    "itemId": item_id, "link": url, "name": name, "brand": brand, "price": price
                })
                logger.info(f"  -> PDP: {item_id}")
            except: continue
        return products

    async def scrape_product_details(self, browser, product_basic, semaphore):
        # Переходит на страницу товара и собирает детальные характеристики (состав, описание)
        async with semaphore:
            item_id = product_basic['itemId']
            url = product_basic['link']
            
            context = await browser.new_context(user_agent=HEADERS['user-agent'])
            page = await context.new_page()
            try:
                await page.goto(url, wait_until="domcontentloaded", timeout=60000)
                
                # Ждем появления ключевого элемента (характеристики)
                # По данным диагностики, иногда требуется до 45 секунд
                try:
                    await page.wait_for_selector("[class*='paired-list__item']", timeout=45000)
                    # Краткий скролл для верности
                    await page.mouse.wheel(0, 800)
                    await asyncio.sleep(1)
                except:
                    logger.warning(f"  [PDP Slow] {item_id} не прогрузился полностью за 45с.")

                html = await page.content()
                return self._parse_details_html(html, product_basic)
            except Exception as e:
                logger.error(f"Error PDP {item_id}: {e}")
                return {**product_basic, "error": str(e)}
            finally:
                await context.close()

    def _get_pdp_tab_text(self, soup, title: str) -> str:
        # Вспомогательный метод для поиска текста во вкладках (Описание, Состав и т.д.)
        tab_div = soup.find('div', attrs={'text': title})
        if tab_div:
            wysiwyg = tab_div.find('div', class_=re.compile(r'wysiwyg'))
            return wysiwyg.get_text(" ", strip=True) if wysiwyg else tab_div.get_text(" ", strip=True)
        
        title_tag = soup.find(string=re.compile(f"^{title}$", re.I))
        if title_tag:
            parent = title_tag.find_parent('div')
            if parent:
                content = parent.find_next_sibling('div', class_=re.compile(r'wysiwyg|content')) or \
                          parent.select_one("[class*='wysiwyg'], [class*='content']")
                if content: return content.get_text(" ", strip=True)
        return ""

    def _parse_pdp_attributes(self, soup) -> dict:
        # Собирает все технические характеристики товара из таблицы "Характеристики"
        attrs = {}
        list_items = soup.select("[class*='paired-list__item']")
        for item in list_items:
            key_tag = item.select_one("[class*='paired-list__key']")
            val_tag = item.select_one("[class*='paired-list__value']")
            if key_tag and val_tag:
                attrs[key_tag.get_text(strip=True).lower()] = val_tag.get_text(strip=True)
        return attrs

    def _parse_details_html(self, raw_html, basic):
        # Главный метод парсинга страницы товара: рейтинг, отзывы и текстовые поля
        result = {k: "" for k in self.fieldnames}
        result.update(basic)
        soup = BeautifulSoup(raw_html, "lxml")
        
        # Rating (Рейтинг)
        # 1. Ищем в специальном классе
        rating_el = soup.select_one("._ga-review-listing-product-rating__rating-value_kvlgk_1") or \
                    soup.select_one("[class*='rating-value']")
        if rating_el:
            result['rating'] = rating_el.get_text(strip=True).replace(',', '.')
        else:
            # 2. Пробуем через мета-теги (Schema.org)
            rating_meta = soup.select_one("meta[itemprop='ratingValue']")
            if rating_meta:
                result['rating'] = rating_meta.get("content", "").replace(',', '.')

        # Review Count (Количество отзывов)
        # 1. Ищем в специальном классе
        rev_count_el = soup.select_one("._ga-review-listing-product-rating__review-count-value_kvlgk_31") or \
                       soup.select_one("[class*='review-count-value']")
        if rev_count_el:
            result['reviews_count'] = "".join(filter(str.isdigit, rev_count_el.get_text()))
        else:
            # 2. Пробуем через мета-теги
            rev_meta = soup.select_one("meta[itemprop='reviewCount']")
            if rev_meta:
                result['reviews_count'] = rev_meta.get("content", "")
            else:
                # 3. Старый способ (поиск текста "отзыв")
                reviews_block = soup.find(string=re.compile(r'\d+\s+отзыв'))
                if reviews_block:
                    result['reviews_count'] = "".join(filter(str.isdigit, reviews_block))
            
        result["description"] = self._get_pdp_tab_text(soup, "Описание")
        result["composition"] = self._get_pdp_tab_text(soup, "Состав")
        result["application"] = self._get_pdp_tab_text(soup, "Применение")
        result["brand_info"] = self._get_pdp_tab_text(soup, "Бренд")

        mapping = {
            "тип продукта": "product_type", "для кого": "for_whom", "назначение": "purpose",
            "тип кожи": "skin_type", "область применения": "area",
            "действующий компонент": "active_ingredient", "время нанесения": "application_time", "объём": "volume"
        }
        attrs = self._parse_pdp_attributes(soup)
        for raw_key, field_name in mapping.items():
            result[field_name] = attrs.get(raw_key, "")
        return result

    def save_batch(self, batch):
        # Сохраняет пачку (batch) собранных товаров в CSV файл
        if not batch: return
        with open(self.output_file, mode="a", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=self.fieldnames, extrasaction="ignore")
            writer.writerows(batch)
        logger.info(f"==> Saved {len(batch)} items to {self.output_file}")

    async def run(self):
        # Основной цикл: проход по страницам каталога и запуск параллельного сбора деталей
        logger.info("=== ULTIMATE SCRAPER START (NO LIMITS MODE) ===")
        async with async_playwright() as p:
            browser = await p.chromium.launch(headless=True)
            total_pages = await self.get_total_pages(browser)
            start_page = 1 # Устанавливаем старт строго с 1-й страницы
            logger.info(f"Total pages to crawl: {total_pages}")
            logger.info(f"Starting from page: {start_page}")
            
            catalog_context = await browser.new_context(user_agent=HEADERS['user-agent'])
            semaphore = asyncio.Semaphore(CONCURRENCY_DETAILS)
            consecutive_empty = 0
            
            for p_no in range(start_page, total_pages + 1):
                page_products = await self.scrape_catalog_page(catalog_context, p_no)
                
                if not page_products:
                    consecutive_empty += 1
                    logger.warning(f"Consecutive empty pages: {consecutive_empty}/{3}")
                    if consecutive_empty >= 3:
                        logger.error("Too many empty pages in a row. Potential block. Exiting for watchdog restart.")
                        await browser.close()
                        sys.exit(1) # Exit to trigger watchdog
                    continue
                
                consecutive_empty = 0 # Reset on success
                tasks = [self.scrape_product_details(browser, pr, semaphore) for pr in page_products]
                results = await asyncio.gather(*tasks)
                self.save_batch([r for r in results if isinstance(r, dict) and "error" not in r])
                await asyncio.sleep(1)
            await browser.close()



## 3. Запуск
Запуск парсера напрямую.

In [ ]:
scraper = GoldAppleUltimateScraper()
await scraper.run() # В Jupyter используем await напрямую


## 4. Watchdog (Авто-рестарт)
Запустите эту ячейку, если хотите, чтобы парсер автоматически перезапускался при блокировках или зависаниях.

In [ ]:
import subprocess
import time
import os
import signal

# Указываем путь к Python внутри виртуального окружения
SCRAPER_COMMAND = ["./.venv/bin/python", "main.py"]
LOG_FILE = "scraper.log"
TRIGGER_PHRASE = "is empty or failed"
MAX_EMPTY_PAGES = 3

def kill_any_scrapers():
    """Принудительно завершает любые запущенные экземпляры скрапера."""
    print("[Watchdog] Cleaning up any existing scraper processes...")
    os.system("pkill -9 -f main.py")
    os.system("pkill -9 -f ultimate_scraper.py")
    os.system("pkill -9 -f chromium")
    time.sleep(2)

def monitor():
    print("--- WATCHDOG STARTED (VENV MODE) ---")
    
    while True:
        kill_any_scrapers()
        print(f"Starting scraper: {' '.join(SCRAPER_COMMAND)}")
        
        # Очищаем лог перед новым циклом
        if os.path.exists(LOG_FILE):
            open(LOG_FILE, 'w').close()
            
        process = subprocess.Popen(SCRAPER_COMMAND)
        
        empty_count = 0
        last_pos = 0
        
        while process.poll() is None:
            if os.path.exists(LOG_FILE):
                with open(LOG_FILE, 'r', encoding='utf-8') as f:
                    f.seek(last_pos)
                    lines = f.readlines()
                    last_pos = f.tell()
                    
                    for line in lines:
                        if TRIGGER_PHRASE in line:
                            empty_count += 1
                            print(f"[Watchdog] Detected empty page! Count: {empty_count}")
                        elif "Saved" in line or "PDP:" in line:
                            # Если нашли товары, сбрасываем счетчик
                            if empty_count > 0:
                                print("[Watchdog] Resetting empty counter because items were found.")
                                empty_count = 0
                            
                if empty_count >= MAX_EMPTY_PAGES:
                    print(f"[Watchdog] TRIGGERED! {empty_count} empty pages. Restarting everything...")
                    # Убиваем всё, включая дочерние процессы playwright
                    kill_any_scrapers()
                    break # Выход для нового старта
            
            time.sleep(2) # Более частая проверка (2 сек)
            
        if process.poll() is not None:
            print("[Watchdog] Scraper process ended or crashed. Restarting in 5s...")
            time.sleep(5)

    monitor()
monitor()
